# Stuttering Detection: k-Nearest Neighbors (KNN) Analysis
**Course**: CS204T (Artificial Intelligence)  
**Team**: 18  
**Focus**: Similarity-based Classification & High-Dimensional Distance

---

## Step 1: Environment Setup
We use the project's modular engine to load high-dimensional WavLM embeddings (768D) and prepare a fair, balanced experimental setup.

In [ ]:
import os
import shutil
import numpy as np
from src.extractors import WavLMExtractor
from src.data import DataManager
from src.models import KNNModel

# Dataset Configuration
AUDIO_DIR = "Stuttering Events in Podcasts Dataset/clips/stuttering-clips/clips"
CSV_PATHS = [
    "Stuttering Events in Podcasts Dataset/SEP-28k_labels.csv",
    "Stuttering Events in Podcasts Dataset/fluencybank_labels.csv"
]
FEATURE_DIR = "data/features"
fluent_dir = os.path.join(FEATURE_DIR, "fluent")
disfluent_dir = os.path.join(FEATURE_DIR, "disfluent")

## Step 2 (Optional): Operational Mode for Data Extraction
Toggle how you want to handle your audio data for this session.
* `SKIP_EXTRACTION`: Uses features already on disk (Default).
* `FORCE_EXTRACT`: Analyzes raw audio for new files (Resumable).
* `CLEAN_START`: Wipes the database and re-extracts from zero.

In [ ]:
# Operational Flags
SKIP_EXTRACTION = True
FORCE_EXTRACT = False
CLEAN_START = False
NUM_CLIPS_TO_EXTRACT = 5

if CLEAN_START:
    if os.path.exists(FEATURE_DIR):
        shutil.rmtree(FEATURE_DIR)
    print("[System] Clean start initiated. Feature database wiped.")

if not SKIP_EXTRACTION or CLEAN_START or FORCE_EXTRACT:
    extractor = WavLMExtractor("microsoft/wavlm-base")
    label_dict = DataManager.generate_label_dict(CSV_PATHS, filter_quality=True)
    sample_wavs = [os.path.join(AUDIO_DIR, f) for f in os.listdir(AUDIO_DIR) 
                   if f.lower().endswith('.wav')][:NUM_CLIPS_TO_EXTRACT]
    extractor.extract_batch(sample_wavs, output_dir=FEATURE_DIR, label_dict=label_dict)
else:
    print("[System] Skipping extraction. Using existing data on disk.")

## Step 3: Data Preparation Engine
We load our distributed features, filter for quality, and use **Oversampling** to handle the 1:3 class imbalance. For KNN specifically, **Standardization** is mandatory, as distance calculations will break if one feature has a higher scale than another.

In [ ]:
label_dict = DataManager.generate_label_dict(CSV_PATHS, filter_quality=True)
manager = DataManager(None, None)

# Loading .npy features
X, y = manager.load_from_folders(fluent_dir, disfluent_dir)
X_train, X_val, X_test, y_train, y_val, y_test = manager.get_splits(test_size=0.15, val_size=0.15)

# Balance Training Set
X_train_bal, y_train_bal = manager.balance_data(X_train, y_train, strategy="oversample")

# KNN Requires scaling to handle Euclidean distance fairy
X_train_final = manager.preprocess(X_train_bal, method="standard")
X_test_final = manager.preprocess(X_test, method="standard")

## Step 4: Model Execution - k-Nearest Neighbors
We instantiate our KNN classifier with $k=5$. This model classifies each speech clip by finding the 5 closest neighbors in Euclidean space and taking a majority vote.

In [ ]:
knn_model = KNNModel("KNN_Baseline_k5", n_neighbors=5)
knn_model.train(X_train_final, y_train_bal)

print("\n--- Evaluation on Unseen Test Set ---")
knn_model.evaluate(X_test_final, y_test)

## Step 5: Research Conclusion - The Curse of Dimensionality
KNN provides an intuitive way to classify clips based on similarity, but its performance on WavLM embeddings (768 features) is capped.

**Finding**: In high-dimensional spaces, the 'distance' between any two samples becomes less meaningful—everyone is approximately equidistant from everyone else. For our project, this explains why models that 'learn' weights (like Neural Networks) achieve higher accuracy than simple neighbor searches.